# MLDA Final Project — AQI Prediction in Taiwan (2024)
*   Ahmad Rifqi Fadhlurrahman - M11402804
*   Gede Darma - M11409803
*   Antonius Yabes Sieman - M11409804
*   Aurelio Naufal Effendy - M11415802

---

**Dataset:** Taiwan EPA Air Quality Monitoring Network — Jan–Aug 2024 (496,092 rows, 85 stations, 22 counties)
**Task:** Regression: predict hourly AQI from pollutant concentrations and meteorological features
**Models:** 11 models:  MLR (baseline), GBR, XGBoost, LightGBM, CatBoost, Random Forest, Stacking, Bagging, Weighted Voting, LSTM, NN (MLP)
**Metrics:** MSE, RMSE, MAE, R², Adjusted R²
**Split:** Temporal 70 / 15 / 15 (no shuffling, strict no-leakage)

---
## 0. Setup & Configuration

All global paths and constants live in `config.py`. Importing it here makes them available across all subsequent cells. We also create the `outputs/` directory structure if it doesn't exist yet.


In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Make sure the project root is on the path so config.py is importable
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import (
    BASE_DIR, DATA_PATH,
    OUTPUT_DIR, FIGURES_DIR, MODELS_DIR, RESULTS_DIR,
    RANDOM_SEED, TRAIN_RATIO, VAL_RATIO, TEST_RATIO,
    OPTUNA_TRIALS, OPTUNA_TIMEOUT, OPTUNA_SUBSAMPLE,
    LOOKBACK, LSTM_EPOCHS, LSTM_BATCH_SIZE, NN_EPOCHS, NN_BATCH_SIZE,
    NUMERIC_FEATURES, TARGET, CAT_FEATURE, POLLUTANT_MAP, POLLUTANT_NONE_LABEL,
)

print(f"Project root : {BASE_DIR}")
print(f"Data file    : {DATA_PATH}")
print(f"Random seed  : {RANDOM_SEED}")
print(f"Split ratios : train={TRAIN_RATIO} / val={VAL_RATIO} / test={TEST_RATIO}")
print(f"Numeric feats: {len(NUMERIC_FEATURES)}")

In [ ]:
# Core imports used throughout the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

print("All imports OK")

---
## 1. Data Preprocessing

**Script:** `01_preprocessing.py`

The preprocessing pipeline performs six steps:
1. **Load & parse** — read the CSV, handle mixed date formats (`YYYY-MM-DD HH:MM` and `YYYY/MM/DD HH:MM:SS`), cast numeric columns from `object` to `float64`
2. **Filter** — remove rows where AQI < 0 (invalid sensor readings)
3. **Pollutant encoding** — map the 6 named categories + NaN to clean labels (`PM25`, `PM10`, `Ozone_8hr`, `Ozone`, `NO2`, `SO2`, `None`), then one-hot encode into 7 binary columns. NaN (~58.6% of rows) is treated as "None" — a valid "no dominant pollutant" state, not missing data
4. **Complete-case deletion** — drop any row with NaN in the 15 numeric features or AQI target (~7.7% of rows)
5. **Temporal split** — sort by date, slice at 70% / 85% / 100% positions. No shuffling ensures no future data leaks into training
6. **StandardScaler** — fit on training set only, transform val and test. This prevents distribution information from val/test influencing the scaling of train features


In [ ]:
from sklearn.preprocessing import StandardScaler

def load_and_clean(path: str) -> pd.DataFrame:
    print("Loading dataset ...")
    df = pd.read_csv(path, low_memory=False)
    print(f"  Raw shape: {df.shape}")

    # Mixed-format date parsing (two formats exist in this dataset)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df['date'] = pd.to_datetime(df['date'], format='mixed')

    # Cast numeric columns (some are loaded as str due to mixed-type CSV)
    for col in NUMERIC_FEATURES + [TARGET]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Remove invalid AQI readings
    before = len(df)
    df = df[df[TARGET] >= 0]
    print(f"  Dropped {before - len(df):,} rows with AQI < 0")

    # Map pollutant labels: NaN -> 'None', clean the category names
    df[CAT_FEATURE] = df[CAT_FEATURE].map(
        lambda x: POLLUTANT_MAP.get(x, POLLUTANT_NONE_LABEL) if pd.notna(x) else POLLUTANT_NONE_LABEL
    )
    return df

In [ ]:
def encode_and_split(df: pd.DataFrame):
    # One-hot encode pollutant — keep all 7 columns (drop_first=False)
    # so every category has its own SHAP value later
    df = pd.get_dummies(df, columns=[CAT_FEATURE], prefix='pol', dtype=float)
    pol_cols    = [c for c in df.columns if c.startswith('pol_')]
    all_features = NUMERIC_FEATURES + pol_cols
    print(f"  OHE pollutant columns ({len(pol_cols)}): {pol_cols}")

    # Complete-case deletion on features + target
    before = len(df)
    df = df.dropna(subset=all_features + [TARGET]).reset_index(drop=True)
    print(f"  Dropped {before - len(df):,} rows (NaN in features/target)")
    print(f"  Clean shape: {df.shape}")

    # Temporal sort then positional slice — NO shuffling
    df = df.sort_values('date').reset_index(drop=True)
    n        = len(df)
    train_end = int(n * TRAIN_RATIO)
    val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    print(f"\n  Temporal split:")
    print(f"    Train : {len(train_df):>7,}  ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
    print(f"    Val   : {len(val_df):>7,}  ({val_df['date'].min().date()} to {val_df['date'].max().date()})")
    print(f"    Test  : {len(test_df):>7,}  ({test_df['date'].min().date()} to {test_df['date'].max().date()})")

    # Fit scaler on train ONLY — then transform val and test
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(train_df[all_features].values)
    X_val   = scaler.transform(val_df[all_features].values)
    X_test  = scaler.transform(test_df[all_features].values)

    y_train = train_df[TARGET].values
    y_val   = val_df[TARGET].values
    y_test  = test_df[TARGET].values

    # Write scaled values back into the DataFrames (needed for LSTM sequence building)
    for i, feat in enumerate(all_features):
        train_df[feat] = X_train[:, i]
        val_df[feat]   = X_val[:, i]
        test_df[feat]  = X_test[:, i]

    return (X_train, y_train, X_val, y_val, X_test, y_test,
            train_df, val_df, test_df, all_features, pol_cols, scaler)

In [ ]:
def run_preprocessing():
    df = load_and_clean(DATA_PATH)

    # Save county/lat/lon for the EDA map before dropping location columns
    map_df = df[['county', 'latitude', 'longitude', TARGET]].dropna().copy()
    map_df[TARGET] = pd.to_numeric(map_df[TARGET], errors='coerce')
    map_df.to_pickle(os.path.join(RESULTS_DIR, 'map_data.pkl'))

    (X_train, y_train, X_val, y_val, X_test, y_test,
     train_df, val_df, test_df, all_features, pol_cols, scaler) = encode_and_split(df)

    joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.pkl'))

    bundle = dict(
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test,
        train_df=train_df, val_df=val_df, test_df=test_df,
        feature_names=all_features, pol_cols=pol_cols,
        n_features=len(all_features),
    )
    joblib.dump(bundle, os.path.join(RESULTS_DIR, 'preprocessed.pkl'))
    print(f"\n  Saved: preprocessed.pkl  |  n_features={len(all_features)}")
    return bundle

bundle = run_preprocessing()
X_train = bundle['X_train'];  y_train = bundle['y_train']
X_val   = bundle['X_val'];    y_val   = bundle['y_val']
X_test  = bundle['X_test'];   y_test  = bundle['y_test']
train_df = bundle['train_df']; val_df = bundle['val_df']; test_df = bundle['test_df']
feature_names = bundle['feature_names']
n_features    = bundle['n_features']
print(f"\nX_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

---
## 2. Exploratory Data Analysis (EDA)

**Script:** `02_eda.py`

Seven plots are generated. All are saved to `outputs/figures/` and displayed inline.

| Plot | Purpose |
|------|---------|
| AQI distribution | Understand target variable shape and central tendency |
| Missing values | Identify which features need attention |
| Correlation heatmap | Find multicollinear features and AQI drivers |
| Feature box-plots | Visualise scale differences and outlier structure |
| AQI by pollutant | Compare AQI across dominant pollutant categories |
| Monthly trend | Reveal seasonal patterns in the 8-month window |
| Taiwan county map | Spatial distribution of average AQI across counties |


In [ ]:
def _save_and_show(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def _load_raw_for_eda() -> pd.DataFrame:
    '''Reload raw CSV with location columns kept for EDA purposes.'''
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        df = pd.read_csv(DATA_PATH, low_memory=False)
        df['date'] = pd.to_datetime(df['date'], format='mixed')
    for col in NUMERIC_FEATURES + [TARGET]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df[df[TARGET].notna() & (df[TARGET] >= 0)]
    df[CAT_FEATURE] = df[CAT_FEATURE].map(
        lambda x: POLLUTANT_MAP.get(x, POLLUTANT_NONE_LABEL) if pd.notna(x) else POLLUTANT_NONE_LABEL
    )
    return df

df_raw = _load_raw_for_eda()
print(f"EDA dataset: {len(df_raw):,} rows")

In [ ]:
# ── Plot 1: AQI Distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw[TARGET], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].axvline(df_raw[TARGET].mean(),   color='red',    linestyle='--', label=f"Mean={df_raw[TARGET].mean():.1f}")
axes[0].axvline(df_raw[TARGET].median(), color='orange', linestyle='--', label=f"Median={df_raw[TARGET].median():.1f}")
axes[0].set_xlabel('AQI'); axes[0].set_ylabel('Frequency')
axes[0].set_title('AQI Distribution (Histogram)'); axes[0].legend()

df_raw[TARGET].plot.kde(ax=axes[1], color='steelblue', linewidth=2)
axes[1].set_xlabel('AQI'); axes[1].set_title('AQI Density (KDE)')

fig.suptitle('AQI Distribution — 2024 Dataset', fontweight='bold', y=1.02)
fig.tight_layout()
_save_and_show(fig, '01_aqi_distribution.png')

In [ ]:
# ── Plot 2: Missing Values ────────────────────────────────────────────────────
# Shows what percentage of each numeric feature is NaN before preprocessing.
# Note: pollutant (~58.6%) is handled separately via OHE — not shown here.
miss = df_raw[NUMERIC_FEATURES + [TARGET]].isnull().mean() * 100
miss = miss[miss > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
miss.plot.bar(ax=ax, color='salmon', edgecolor='white')
ax.set_ylabel('Missing (%)'); ax.set_title('Missing Value Rate per Feature', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
_save_and_show(fig, '02_missing_values.png')

In [ ]:
# ── Plot 3: Pearson Correlation Heatmap ──────────────────────────────────────
# Full lower-triangle heatmap — helps identify multicollinear feature pairs
# and which features are most strongly correlated with AQI.
corr = df_raw[NUMERIC_FEATURES + [TARGET]].corr(method='pearson')

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', annot_kws={'size': 7},
            cmap='coolwarm', center=0, vmin=-1, vmax=1, linewidths=0.3, square=True)
ax.set_title('Pearson Correlation Heatmap', fontweight='bold', pad=12)
fig.tight_layout()
_save_and_show(fig, '03_correlation_heatmap.png')

In [ ]:
# ── Plot 4: Feature Box-Plots ─────────────────────────────────────────────────
# Shows the scale and outlier structure of each numeric feature.
# Large scale differences (e.g. winddirec 0-360 vs co 0-5) confirm
# that StandardScaler is necessary before model training.
ncols = 4
nrows = (len(NUMERIC_FEATURES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    data = df_raw[col].dropna()
    axes[i].boxplot(data, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='steelblue'),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', markersize=1, alpha=0.3))
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(axis='x', bottom=False, labelbottom=False)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Box-Plots (Scale & Outlier Overview)', fontweight='bold', y=1.01)
fig.tight_layout()
_save_and_show(fig, '04_feature_boxplots.png')

In [ ]:
# ── Plot 5: AQI by Pollutant Category ────────────────────────────────────────
# Groups records by their dominant pollutant and compares AQI distributions.
# PM2.5-dominated hours have the highest AQI; "None" (no dominant pollutant)
# corresponds to the cleanest-air periods.
order = df_raw.groupby(CAT_FEATURE)[TARGET].median().sort_values(ascending=False).index
colors = plt.cm.Set2(np.linspace(0, 1, len(order)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
data_by_pol = [df_raw[df_raw[CAT_FEATURE] == cat][TARGET].dropna().values for cat in order]
bp = axes[0].boxplot(data_by_pol, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_xticks(range(1, len(order) + 1))
axes[0].set_xticklabels(order, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('AQI'); axes[0].set_title('AQI by Dominant Pollutant')

means = df_raw.groupby(CAT_FEATURE)[TARGET].mean().reindex(order)
stds  = df_raw.groupby(CAT_FEATURE)[TARGET].std().reindex(order)
axes[1].bar(range(len(order)), means, yerr=stds, capsize=4, color=colors, edgecolor='white')
axes[1].set_xticks(range(len(order)))
axes[1].set_xticklabels(order, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('Mean AQI +/- std'); axes[1].set_title('Mean AQI per Pollutant Category')

fig.suptitle('AQI by Dominant Pollutant Category', fontweight='bold')
fig.tight_layout()
_save_and_show(fig, '05_aqi_by_pollutant.png')

In [ ]:
# ── Plot 6: Monthly AQI Trend ─────────────────────────────────────────────────
# Aggregates hourly readings to monthly mean/median and plots a trend line.
# Shows AQI peaks in winter months (Jan-Feb) and a gradual decline toward
# the summer monsoon season (Jul-Aug).
df_tmp = df_raw.copy()
df_tmp['month'] = df_tmp['date'].dt.to_period('M')
monthly = df_tmp.groupby('month')[TARGET].agg(['mean', 'std', 'median']).reset_index()
monthly['month_str'] = monthly['month'].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(monthly['month_str'],
                monthly['mean'] - monthly['std'],
                monthly['mean'] + monthly['std'],
                alpha=0.2, color='steelblue', label='+/- 1 std')
ax.plot(monthly['month_str'], monthly['mean'],   'o-',  color='steelblue', label='Mean AQI',   linewidth=2)
ax.plot(monthly['month_str'], monthly['median'], 's--', color='orange',    label='Median AQI', linewidth=1.5)
ax.set_xlabel('Month'); ax.set_ylabel('AQI')
ax.set_title('Monthly AQI Trend -- Jan to Aug 2024', fontweight='bold')
ax.legend(); ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
_save_and_show(fig, '06_monthly_aqi_trend.png')

In [ ]:
# ── Plot 7: Taiwan County Choropleth Map ──────────────────────────────────────
# Uses the official Taiwan county shapefile (Taiwanmap_shp/tw.shp) to draw a
# proper polygon choropleth. Average AQI per county is computed from the raw
# dataset and merged with the GeoDataFrame. Color scale: green (low) -> red (high).
import geopandas as gpd

SHP_TO_COUNTY = {
    'Kinmen': 'Kinmen County', 'Matsu Islands': 'Lienchiang County',
    'Penghu': 'Penghu County', 'Taoyuan': 'Taoyuan City',
    'Hsinchu': 'Hsinchu County', 'Hsinchu City': 'Hsinchu City',
    'Miaoli': 'Miaoli County', 'Taichung City': 'Taichung City',
    'Changhua': 'Changhua County', 'Yunlin': 'Yunlin County',
    'Chiayi': 'Chiayi County', 'Chiayi City': 'Chiayi City',
    'Tainan City': 'Tainan City', 'Kaohsiung City': 'Kaohsiung City',
    'Pingtung': 'Pingtung County', 'Taitung': 'Taitung County',
    'Hualien': 'Hualien County', 'Yilan': 'Yilan County',
    'New Taipei City': 'New Taipei City', 'Keelung City': 'Keelung City',
    'Nantou': 'Nantou County', 'Taipei City': 'Taipei City',
}

map_df = pd.read_pickle(os.path.join(RESULTS_DIR, 'map_data.pkl'))
county_stats = (map_df.groupby('county')
                .agg(avg_aqi=(TARGET, 'mean'), n=(TARGET, 'count'))
                .dropna().reset_index())

gdf = gpd.read_file(os.path.join(BASE_DIR, 'Taiwanmap_shp', 'tw.shp'))
gdf['county'] = gdf['name'].map(SHP_TO_COUNTY)
gdf = gdf.merge(county_stats[['county', 'avg_aqi']], on='county', how='left')

vmin, vmax = gdf['avg_aqi'].min(), gdf['avg_aqi'].max()
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.RdYlGn_r

fig, ax = plt.subplots(figsize=(9, 11))
ax.set_facecolor('#a8d8ea')
gdf.plot(column='avg_aqi', cmap=cmap, norm=norm, linewidth=0.6,
         edgecolor='#444444', ax=ax,
         missing_kwds={'color': '#cccccc', 'edgecolor': '#999999', 'label': 'No data'})
for _, row in gdf.iterrows():
    if row.geometry is None: continue
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    label = (row['county'] if pd.notna(row['county']) else row['name'])
    label = label.replace(' County', ' Co.').replace(' City', ' City')
    ax.annotate(label, xy=(cx, cy), ha='center', va='center',
                fontsize=5.8, fontweight='bold', color='#1a1a1a',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.55, lw=0))
ax.set_xlim(118.8, 122.8); ax.set_ylim(21.8, 25.7)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Average AQI by County -- Taiwan, Jan-Aug 2024', fontsize=13, fontweight='bold', pad=14)
ax.grid(True, alpha=0.2, linestyle='--')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.02).set_label('Avg AQI')
fig.tight_layout()
_save_and_show(fig, '07_taiwan_map.png')

---
## 3. Model Training

**Script:** `03_models.py`

### Utility functions

Before the individual models, three shared utilities are defined:
- **`adj_r2`** — computes Adjusted R² penalising for number of features
- **`run_cv`** — runs 5-fold `TimeSeriesSplit` cross-validation using `cross_val_score`, maintaining temporal order within each fold
- **`subsample`** — draws a random 25% subset of training data used during Optuna trials to speed up each trial

### Hyperparameter tuning with Optuna

For GBR, XGBoost, LightGBM, CatBoost, and Random Forest, an Optuna study is created with the **TPE (Tree-structured Parzen Estimator) sampler**. Each trial:
1. Trains the model on the 25% subsample of training data
2. Evaluates MSE on the full validation set
3. Returns MSE as the objective to minimise

After 30 trials (or 360-second timeout), the best hyperparameters are used to retrain on the **full** training set.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

def adj_r2(r2: float, n: int, k: int) -> float:
    '''Adjusted R-squared - penalises model complexity by number of features k.'''
    if n <= k + 1:
        return float('nan')
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)

def run_cv(model, X_tr, y_tr, n_feat: int) -> dict:
    '''
    5-fold TimeSeriesSplit CV on training data.
    Returns mean and std of RMSE and R2 across folds.
    TimeSeriesSplit is used instead of KFold to preserve temporal ordering:
    each fold training window is always earlier than its validation window.
    '''
    tscv = TimeSeriesSplit(n_splits=5)
    rmse = -cross_val_score(model, X_tr, y_tr,
                            cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    r2   =  cross_val_score(model, X_tr, y_tr,
                            cv=tscv, scoring='r2', n_jobs=-1)
    return dict(cv_rmse_mean=rmse.mean(), cv_rmse_std=rmse.std(),
                cv_r2_mean=r2.mean(),    cv_r2_std=r2.std())

def subsample(X, y, frac=OPTUNA_SUBSAMPLE, seed=RANDOM_SEED):
    '''Draw a sorted random subset for faster Optuna trials.'''
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=int(len(X) * frac), replace=False)
    idx.sort()
    return X[idx], y[idx]

def _save_model(model, name: str):
    joblib.dump(model, os.path.join(MODELS_DIR, f'{name}.pkl'))

predictions = {}   # will accumulate {model_name: {val_pred, test_pred, val_true, test_true, cv_*}}

def _record(name, val_pred, test_pred, cv_res, val_true=None, test_true=None):
    predictions[name] = dict(
        val_pred=val_pred, test_pred=test_pred,
        val_true=val_true if val_true is not None else y_val,
        test_true=test_true if test_true is not None else y_test,
        **cv_res,
    )

print("Utilities defined.")

### Model 1 — Multiple Linear Regression (Baseline)

OLS regression with no regularisation. Serves as the performance floor: if a model can't beat MLR, it's not worth using. The relatively low R² (~0.89) of MLR compared to tree models confirms the non-linear nature of AQI dynamics.


In [ ]:
from sklearn.linear_model import LinearRegression

print("[1/11] Multiple Linear Regression ...")
mlr = LinearRegression(n_jobs=-1)
mlr.fit(X_train, y_train)

vp = mlr.predict(X_val);  tp = mlr.predict(X_test)
cv = run_cv(LinearRegression(n_jobs=-1), X_train, y_train, n_features)
_record('MLR', vp, tp, cv)
_save_model(mlr, 'mlr')
print(f"  Val R2 : {r2_score(y_val, vp):.4f}")
print(f"  Test R2: {r2_score(y_test, tp):.4f}")
print(f"  CV R2  : {cv['cv_r2_mean']:.4f} +/- {cv['cv_r2_std']:.4f}")

### Model 2 — Gradient Boosting Regressor (GBR)

Sklearn's `GradientBoostingRegressor` builds trees sequentially, each fitting the **residual errors** of the previous. Optuna searches over learning rate (log scale), tree depth, number of estimators, subsample fraction, and minimum leaf samples.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

def tune_gbr(X_sub, y_sub):
    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 50, 300),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            max_depth        = trial.suggest_int('max_depth', 3, 7),
            min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20),
            subsample        = trial.suggest_float('subsample', 0.5, 1.0),
            random_state     = RANDOM_SEED,
        )
        m = GradientBoostingRegressor(**params)
        m.fit(X_sub, y_sub)
        return mean_squared_error(y_val, m.predict(X_val))
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT)
    return study.best_params

print("[2/11] GBR (Optuna) ...")
X_sub, y_sub = subsample(X_train, y_train)
best_gbr = tune_gbr(X_sub, y_sub);  best_gbr['random_state'] = RANDOM_SEED
print(f"  Best params: {best_gbr}")
gbr = GradientBoostingRegressor(**best_gbr)
gbr.fit(X_train, y_train)
vp = gbr.predict(X_val);  tp = gbr.predict(X_test)
cv = run_cv(GradientBoostingRegressor(**best_gbr), X_train, y_train, n_features)
_record('GBR', vp, tp, cv);  _save_model(gbr, 'gbr')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 3 — XGBoost

Adds L1/L2 regularisation (`reg_alpha`, `reg_lambda`) and column subsampling (`colsample_bytree`) on top of gradient boosting, making it more robust against overfitting on high-dimensional data. Early stopping is applied during Optuna trials using the validation set.


In [ ]:
import xgboost as xgb

def tune_xgb(X_sub, y_sub):
    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 50, 300),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            max_depth        = trial.suggest_int('max_depth', 3, 8),
            subsample        = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
            reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
            random_state=RANDOM_SEED, verbosity=0, n_jobs=-1,
        )
        m = xgb.XGBRegressor(**params)
        m.fit(X_sub, y_sub, eval_set=[(X_val, y_val)], verbose=False)
        return mean_squared_error(y_val, m.predict(X_val))
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT)
    return study.best_params

print("[3/11] XGBoost (Optuna) ...")
X_sub, y_sub = subsample(X_train, y_train)
best_xgb = tune_xgb(X_sub, y_sub);  best_xgb.update({'random_state': RANDOM_SEED, 'verbosity': 0, 'n_jobs': -1})
print(f"  Best params: {best_xgb}")
xgb_m = xgb.XGBRegressor(**best_xgb)
xgb_m.fit(X_train, y_train)
vp = xgb_m.predict(X_val);  tp = xgb_m.predict(X_test)
cv = run_cv(xgb.XGBRegressor(**best_xgb), X_train, y_train, n_features)
_record('XGBoost', vp, tp, cv);  _save_model(xgb_m, 'xgb')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 4 — LightGBM

Uses **leaf-wise (best-first) tree growth** and histogram-based binning, making it significantly faster than XGBoost on datasets with many rows. `num_leaves` is a key parameter controlling model complexity. Early stopping (patience=20 rounds) is applied during Optuna trials.


In [ ]:
import lightgbm as lgb

def tune_lgb(X_sub, y_sub):
    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 50, 400),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            max_depth        = trial.suggest_int('max_depth', 3, 8),
            num_leaves       = trial.suggest_int('num_leaves', 15, 127),
            min_child_samples= trial.suggest_int('min_child_samples', 5, 50),
            subsample        = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
        )
        m = lgb.LGBMRegressor(**params)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            m.fit(X_sub, y_sub, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)])
        return mean_squared_error(y_val, m.predict(X_val))
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT)
    return study.best_params

print("[4/11] LightGBM (Optuna) ...")
X_sub, y_sub = subsample(X_train, y_train)
best_lgb = tune_lgb(X_sub, y_sub);  best_lgb.update({'random_state': RANDOM_SEED, 'n_jobs': -1, 'verbose': -1})
print(f"  Best params: {best_lgb}")
lgb_m = lgb.LGBMRegressor(**best_lgb)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    lgb_m.fit(X_train, y_train)
vp = lgb_m.predict(X_val);  tp = lgb_m.predict(X_test)
cv = run_cv(lgb.LGBMRegressor(**best_lgb), X_train, y_train, n_features)
_record('LightGBM', vp, tp, cv);  _save_model(lgb_m, 'lgb')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 5 — CatBoost

Uses **ordered boosting** and symmetric trees. `use_best_model=True` during each Optuna trial ensures the trial evaluates the checkpoint with the lowest validation loss, similar to early stopping.


In [ ]:
from catboost import CatBoostRegressor

def tune_cb(X_sub, y_sub):
    def objective(trial):
        params = dict(
            iterations    = trial.suggest_int('iterations', 50, 300),
            learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            depth         = trial.suggest_int('depth', 3, 8),
            l2_leaf_reg   = trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
            random_seed=RANDOM_SEED, silent=True,
        )
        m = CatBoostRegressor(**params)
        m.fit(X_sub, y_sub, eval_set=(X_val, y_val), use_best_model=True)
        return mean_squared_error(y_val, m.predict(X_val))
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT)
    return study.best_params

print("[5/11] CatBoost (Optuna) ...")
X_sub, y_sub = subsample(X_train, y_train)
best_cb = tune_cb(X_sub, y_sub);  best_cb.update({'random_seed': RANDOM_SEED, 'silent': True})
print(f"  Best params: {best_cb}")
cb_m = CatBoostRegressor(**best_cb)
cb_m.fit(X_train, y_train)
vp = cb_m.predict(X_val);  tp = cb_m.predict(X_test)
cv = run_cv(CatBoostRegressor(**best_cb), X_train, y_train, n_features)
_record('CatBoost', vp, tp, cv);  _save_model(cb_m, 'cb')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 6 — Random Forest

Unlike boosting, trees are grown **independently** on bootstrap samples and averaged. No sequential error correction — variance reduction through diversity. `max_features` controls how many features each tree considers at each split.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

def tune_rf(X_sub, y_sub):
    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 50, 300),
            max_depth        = trial.suggest_int('max_depth', 4, 20),
            min_samples_split= trial.suggest_int('min_samples_split', 2, 20),
            min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10),
            max_features     = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
            random_state=RANDOM_SEED, n_jobs=-1,
        )
        m = RandomForestRegressor(**params)
        m.fit(X_sub, y_sub)
        return mean_squared_error(y_val, m.predict(X_val))
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT)
    return study.best_params

print("[6/11] Random Forest (Optuna) ...")
X_sub, y_sub = subsample(X_train, y_train)
best_rf = tune_rf(X_sub, y_sub);  best_rf.update({'random_state': RANDOM_SEED, 'n_jobs': -1})
print(f"  Best params: {best_rf}")
rf_m = RandomForestRegressor(**best_rf)
rf_m.fit(X_train, y_train)
vp = rf_m.predict(X_val);  tp = rf_m.predict(X_test)
cv = run_cv(RandomForestRegressor(**best_rf), X_train, y_train, n_features)
_record('RandomForest', vp, tp, cv);  _save_model(rf_m, 'rf')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 7 — Stacking Ensemble

Stacking uses **out-of-fold (OOF) predictions** from four base learners (GBR, XGB, LGB, CB) as features for a Ridge regression meta-learner.

> **Important:** `StackingRegressor` calls `cross_val_predict` internally, which requires every training sample to appear in **exactly one** test fold (partition constraint). `TimeSeriesSplit` violates this — early samples never land in any test fold. So `cv=5` (KFold) is used inside the StackingRegressor, while the outer CV for robustness evaluation still uses `TimeSeriesSplit`.

The nested CV (5 outer × 5 inner × 4 models = 100 fits) takes approximately 40 minutes.


In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

print("[7/11] Stacking Ensemble ...")
estimators = [('gbr', gbr), ('xgb', xgb_m), ('lgb', lgb_m), ('cb', cb_m)]
stk = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(),
    cv=5,          # KFold required — TimeSeriesSplit violates cross_val_predict partition constraint
    n_jobs=-1, passthrough=False,
)
stk.fit(X_train, y_train)
vp = stk.predict(X_val);  tp = stk.predict(X_test)

print("  Running nested 5-fold CV (slow: ~40 min) ...")
cv = run_cv(StackingRegressor(estimators=estimators, final_estimator=Ridge(), cv=5, n_jobs=-1),
            X_train, y_train, n_features)
_record('Stacking', vp, tp, cv);  _save_model(stk, 'stacking')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 8 — Bagging Ensemble

`BaggingRegressor` trains 100 independent `DecisionTreeRegressor` (depth 12) models on bootstrap samples (80% of rows and 80% of features per bag). Predictions are averaged. Pure variance reduction — no error correction between trees.


In [ ]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

print("[8/11] Bagging Ensemble ...")
bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=12, random_state=RANDOM_SEED),
    n_estimators=100, max_samples=0.8, max_features=0.8,
    bootstrap=True, random_state=RANDOM_SEED, n_jobs=-1,
)
bag.fit(X_train, y_train)
vp = bag.predict(X_val);  tp = bag.predict(X_test)
cv = run_cv(BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=12, random_state=RANDOM_SEED),
        n_estimators=50, max_samples=0.8, max_features=0.8,
        bootstrap=True, random_state=RANDOM_SEED, n_jobs=-1),
    X_train, y_train, n_features)
_record('Bagging', vp, tp, cv);  _save_model(bag, 'bagging')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 9 — Weighted Voting Ensemble

`VotingRegressor` combines the four tuned boosting models with weights reflecting their individual validation performance: **GBR×4, CatBoost×3, XGBoost×2, LightGBM×1**. The final prediction is a weighted average — no refitting, no meta-learner. This mirrors the proposed ensemble in the reference paper.


In [ ]:
from sklearn.ensemble import VotingRegressor

print("[9/11] Weighted Voting Ensemble ...")
vot = VotingRegressor(
    estimators=[('gbr', gbr), ('cb', cb_m), ('xgb', xgb_m), ('lgb', lgb_m)],
    weights=[4, 3, 2, 1], n_jobs=-1,
)
vot.fit(X_train, y_train)
vp = vot.predict(X_val);  tp = vot.predict(X_test)
cv = run_cv(VotingRegressor(
        estimators=[('gbr', gbr), ('cb', cb_m), ('xgb', xgb_m), ('lgb', lgb_m)],
        weights=[4, 3, 2, 1], n_jobs=-1),
    X_train, y_train, n_features)
_record('WeightedVoting', vp, tp, cv);  _save_model(vot, 'voting')
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}  CV R2={cv['cv_r2_mean']:.4f}+/-{cv['cv_r2_std']:.4f}")

### Model 10 — LSTM

Sequences of length 24 (24-hour lookback) are constructed **per monitoring station** to avoid mixing readings from different stations at sequence boundaries. All sequences are then pooled into a single training batch.

Architecture: `LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(16, ReLU) → Dense(1)`

> **Note:** 5-fold CV is not performed for LSTM — Keras models are not scikit-learn estimators. Learning curves (train vs val loss) are used instead to assess overfitting.


In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)

def _make_sequences(df_split, feat_cols, target_col, lookback):
    '''Build (n_samples, lookback, n_features) sequences per station, then pool all.'''
    Xs, ys = [], []
    for _, grp in df_split.groupby('siteid', sort=False):
        grp = grp.sort_values('date')
        X = grp[feat_cols].values.astype(np.float32)
        y = grp[target_col].values.astype(np.float32)
        for i in range(lookback, len(grp)):
            Xs.append(X[i - lookback:i])
            ys.append(y[i])
    if not Xs:
        return np.empty((0, lookback, len(feat_cols))), np.empty(0)
    return np.stack(Xs), np.array(ys)

print("[10/11] LSTM ...")
X_tr_seq, y_tr_seq = _make_sequences(train_df, feature_names, TARGET, LOOKBACK)
X_vl_seq, y_vl_seq = _make_sequences(val_df,   feature_names, TARGET, LOOKBACK)
X_ts_seq, y_ts_seq = _make_sequences(test_df,  feature_names, TARGET, LOOKBACK)
print(f"  Sequences: train={X_tr_seq.shape}  val={X_vl_seq.shape}  test={X_ts_seq.shape}")

inp  = tf.keras.Input(shape=(LOOKBACK, n_features))
x    = tf.keras.layers.LSTM(64, return_sequences=True)(inp)
x    = tf.keras.layers.Dropout(0.2)(x)
x    = tf.keras.layers.LSTM(32)(x)
x    = tf.keras.layers.Dropout(0.2)(x)
x    = tf.keras.layers.Dense(16, activation='relu')(x)
out  = tf.keras.layers.Dense(1)(x)
lstm = tf.keras.Model(inp, out)
lstm.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
lstm.summary()

history_lstm = lstm.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_vl_seq, y_vl_seq),
    epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=0),
    ],
    verbose=1,
)
vp = lstm.predict(X_vl_seq, verbose=0).flatten()
tp = lstm.predict(X_ts_seq, verbose=0).flatten()
cv_nan = dict(cv_rmse_mean=float('nan'), cv_rmse_std=float('nan'),
              cv_r2_mean=float('nan'),   cv_r2_std=float('nan'))
_record('LSTM', vp, tp, cv_nan, val_true=y_vl_seq, test_true=y_ts_seq)
lstm.save(os.path.join(MODELS_DIR, 'lstm.keras'))
joblib.dump(history_lstm.history, os.path.join(RESULTS_DIR, 'lstm_history.pkl'))
print(f"  Val R2={r2_score(y_vl_seq, vp):.4f}  Test R2={r2_score(y_ts_seq, tp):.4f}")

### Model 11 — Neural Network (MLP)

A simple 3-hidden-layer feed-forward network on the flat 22-feature vector. No temporal window — each row is treated independently. Dropout layers prevent overfitting. Same training protocol as LSTM (early stopping + ReduceLROnPlateau).


In [ ]:
print("[11/11] Neural Network (MLP) ...")
inp  = tf.keras.Input(shape=(n_features,))
x    = tf.keras.layers.Dense(128, activation='relu')(inp)
x    = tf.keras.layers.Dropout(0.3)(x)
x    = tf.keras.layers.Dense(64,  activation='relu')(x)
x    = tf.keras.layers.Dropout(0.3)(x)
x    = tf.keras.layers.Dense(32,  activation='relu')(x)
out  = tf.keras.layers.Dense(1)(x)
nn   = tf.keras.Model(inp, out)
nn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
nn.summary()

history_nn = nn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=NN_EPOCHS, batch_size=NN_BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=0),
    ],
    verbose=1,
)
vp = nn.predict(X_val,  verbose=0).flatten()
tp = nn.predict(X_test, verbose=0).flatten()
_record('NN', vp, tp, cv_nan)
nn.save(os.path.join(MODELS_DIR, 'nn.keras'))
joblib.dump(history_nn.history, os.path.join(RESULTS_DIR, 'nn_history.pkl'))
print(f"  Val R2={r2_score(y_val, vp):.4f}  Test R2={r2_score(y_test, tp):.4f}")

# Persist all predictions for evaluation step
joblib.dump(predictions, os.path.join(RESULTS_DIR, 'predictions.pkl'))
print("\nAll predictions saved.")

---
## 4. Evaluation

**Script:** `04_evaluation.py`

Five metrics are computed for every model on both the **validation** and **test** sets:
- **MSE** — mean squared error (primary, sensitive to large errors)
- **RMSE** — root MSE (same unit as AQI, easier to interpret)
- **MAE** — mean absolute error (robust to outliers)
- **R²** — proportion of variance explained
- **Adjusted R²** — R² penalised for number of features (k=22): `1 - (1-R²)(n-1)/(n-k-1)`

For robustness, **5-fold TimeSeriesSplit CV** mean ± std of RMSE and R² are also reported (NaN for deep learning models).


In [ ]:
MODEL_ORDER = ['MLR','GBR','XGBoost','LightGBM','CatBoost',
               'RandomForest','Stacking','Bagging','WeightedVoting','LSTM','NN']

def compute_metrics_row(y_true, y_pred, n_feat, split, model):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    ar2  = adj_r2(r2, len(y_true), n_feat)
    return {'Model': model, 'Split': split,
            'MSE': round(mse,4), 'RMSE': round(rmse,4), 'MAE': round(mae,4),
            'R2': round(r2,4), 'Adj_R2': round(ar2,4)}

# Reload predictions in case we're running evaluation after a restart
predictions = joblib.load(os.path.join(RESULTS_DIR, 'predictions.pkl'))

rows = []
for model_name in MODEL_ORDER:
    if model_name not in predictions:
        continue
    p = predictions[model_name]
    rows.append(compute_metrics_row(p['val_true'],  p['val_pred'],  n_features, 'Val',  model_name))
    rows.append(compute_metrics_row(p['test_true'], p['test_pred'], n_features, 'Test', model_name))

df_metrics = pd.DataFrame(rows)

cv_rows = []
for m in MODEL_ORDER:
    if m not in predictions: continue
    p = predictions[m]
    cv_rows.append({'Model': m,
                    'CV_RMSE_mean': round(p.get('cv_rmse_mean', float('nan')), 4),
                    'CV_RMSE_std':  round(p.get('cv_rmse_std',  float('nan')), 4),
                    'CV_R2_mean':   round(p.get('cv_r2_mean',   float('nan')), 4),
                    'CV_R2_std':    round(p.get('cv_r2_std',    float('nan')), 4)})
df_cv = pd.DataFrame(cv_rows)

df_all = df_metrics.merge(df_cv, on='Model', how='left')
df_all.to_csv(os.path.join(RESULTS_DIR, 'metrics_table.csv'), index=False)

print("=== VALIDATION SET ===")
display(df_all[df_all['Split']=='Val'].drop(columns='Split').reset_index(drop=True))
print("\n=== TEST SET ===")
display(df_all[df_all['Split']=='Test'].drop(columns='Split').reset_index(drop=True))
print("\n=== 5-FOLD CV (Training Set) ===")
display(df_cv)

In [ ]:
# ── Metrics Comparison Bar Chart ──────────────────────────────────────────────
# Each panel is sorted from worst (top) to best (bottom) for quick ranking.
metrics = ['MSE','RMSE','MAE','R2','Adj_R2']
lower_is_better = {'MSE','RMSE','MAE'}

fig, axes = plt.subplots(len(metrics), 2, figsize=(16, 4*len(metrics)))
for ri, metric in enumerate(metrics):
    for ci, split in enumerate(['Val','Test']):
        ax  = axes[ri][ci]
        sub = df_all[df_all['Split']==split][['Model',metric]].dropna(subset=[metric])
        sub = sub.sort_values(metric, ascending=(metric not in lower_is_better))
        vals = sub[metric].values;  models = sub['Model'].values
        bars = ax.barh(models, vals, color='#2980b9', edgecolor='white', alpha=0.85)
        ax.set_title(f'{metric} -- {split}', fontsize=9, fontweight='bold')
        x_pad = (vals.max()-vals.min())*0.01 if vals.max()!=vals.min() else vals.max()*0.01
        for bar, v in zip(bars, vals):
            ax.text(bar.get_width()+x_pad, bar.get_y()+bar.get_height()/2,
                    f'{v:.4f}', va='center', fontsize=7)
        ax.invert_yaxis()
fig.suptitle('Model Performance Comparison -- All Metrics (sorted by score)',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
_save_and_show(fig, '08_metrics_comparison.png')

In [ ]:
# ── CV Robustness Chart ───────────────────────────────────────────────────────
df_cv_plot = df_cv.dropna(subset=['CV_RMSE_mean'])
x = np.arange(len(df_cv_plot))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x, df_cv_plot['CV_RMSE_mean'], yerr=df_cv_plot['CV_RMSE_std'],
            capsize=4, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(df_cv_plot['Model'], rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('RMSE'); axes[0].set_title('5-Fold CV -- RMSE (mean +/- std)')
axes[1].bar(x, df_cv_plot['CV_R2_mean'], yerr=df_cv_plot['CV_R2_std'],
            capsize=4, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(df_cv_plot['Model'], rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('R2'); axes[1].set_title('5-Fold CV -- R2 (mean +/- std)')
axes[1].set_ylim(max(0, df_cv_plot['CV_R2_mean'].min()-0.05), 1.01)
fig.suptitle('Cross-Validation Robustness (TimeSeriesSplit, k=5)', fontweight='bold')
fig.tight_layout()
_save_and_show(fig, '09_cv_robustness.png')

In [ ]:
# ── Predicted vs Actual + Residuals (best model by Test R2) ──────────────────
best_name = df_all[df_all['Split']=='Test'].loc[lambda x: x['R2'].idxmax(), 'Model']
print(f"Best model by Test R2: {best_name}")
p = predictions[best_name]
rng = np.random.default_rng(42)
idx = rng.choice(len(p['test_true']), size=min(5000, len(p['test_true'])), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(p['test_true'][idx], p['test_pred'][idx], alpha=0.25, s=8, color='steelblue')
lims = [min(p['test_true'].min(), p['test_pred'].min()), max(p['test_true'].max(), p['test_pred'].max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual AQI'); axes[0].set_ylabel('Predicted AQI')
axes[0].set_title(f'{best_name} -- Predicted vs Actual (Test)'); axes[0].legend()

residuals = p['test_true'] - p['test_pred']
axes[1].hist(residuals, bins=60, color='salmon', edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residual (Actual - Predicted)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'{best_name} -- Residual Distribution')
fig.suptitle(f'Best Model: {best_name}', fontweight='bold')
fig.tight_layout()
_save_and_show(fig, '10_pred_vs_actual_best.png')

In [ ]:
# ── Learning Curves (LSTM and NN) ─────────────────────────────────────────────
# For deep learning, learning curves replace CV as the overfitting diagnostic.
# If val_loss diverges from train_loss, the model is overfitting.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name, pkl in [(axes[0],'LSTM','lstm_history.pkl'), (axes[1],'NN (MLP)','nn_history.pkl')]:
    path = os.path.join(RESULTS_DIR, pkl)
    if not os.path.exists(path):
        ax.set_title(f'{name} -- history not found'); continue
    h = joblib.load(path)
    epochs = range(1, len(h['loss'])+1)
    ax.plot(epochs, h['loss'],     label='Train loss', color='steelblue')
    ax.plot(epochs, h['val_loss'], label='Val loss',   color='orange', linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.set_title(f'{name} -- Learning Curve'); ax.legend()
fig.suptitle('Deep Learning Learning Curves', fontweight='bold')
fig.tight_layout()
_save_and_show(fig, '11_learning_curves.png')

---
## 5. SHAP Interpretability

**Script:** `05_shap_analysis.py`

**SHAP (SHapley Additive exPlanations)** assigns each feature a contribution value for each prediction, based on cooperative game theory. For tree-based models, `shap.TreeExplainer` computes exact Shapley values efficiently.

Three plot types per model:
- **Beeswarm summary** — magnitude + direction of each feature's effect across all samples (colour = feature value)
- **Bar importance** — mean |SHAP| across samples = average contribution magnitude
- **Dependence plots** — how SHAP value of the top-3 features changes across their value range

SHAP is applied to:
1. The **best model** (GBR by Test R²)
2. The **WeightedVoting ensemble** (as reference — uses a model-agnostic fallback if TreeExplainer fails)
3. All five tree models for a **comparative importance chart**

A random sample of 3,000 test rows is used for speed.


In [ ]:
import shap

SHAP_SAMPLE = 3000

def run_shap(model, model_name, X_test, feature_names, tag):
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X_test), size=min(SHAP_SAMPLE, len(X_test)), replace=False)
    X_bg = X_test[idx]
    feat_names = np.array(feature_names)

    print(f"  Building explainer for {model_name} ...")
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        # TreeExplainer works for single-algorithm tree models
        if model_name in ('GBR','XGBoost','LightGBM','CatBoost','RandomForest','Bagging'):
            try:
                explainer   = shap.TreeExplainer(model)
                shap_values = explainer(X_bg)
            except Exception:
                explainer   = shap.Explainer(model.predict, shap.maskers.Independent(X_bg, max_samples=500))
                shap_values = explainer(X_bg)
        else:
            # VotingRegressor / StackingRegressor: use model-agnostic KernelExplainer fallback
            explainer   = shap.Explainer(model.predict, shap.maskers.Independent(X_bg, max_samples=500))
            shap_values = explainer(X_bg)

    vals = shap_values.values
    mean_abs = np.abs(vals).mean(axis=0)
    order    = np.argsort(mean_abs)[::-1]

    # Beeswarm
    fig, _ = plt.subplots(figsize=(10, 8))
    shap.summary_plot(vals, X_bg, feature_names=feat_names, show=False, max_display=20)
    plt.title(f'SHAP Beeswarm -- {model_name}', fontweight='bold', pad=10)
    _save_and_show(plt.gcf(), f'12_shap_summary_{tag}.png')

    # Bar importance
    fig, ax = plt.subplots(figsize=(9, 6))
    top20 = order[:20]
    ax.barh(feat_names[top20][::-1], mean_abs[top20][::-1], color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'Feature Importance (Mean |SHAP|) -- {model_name}', fontweight='bold')
    fig.tight_layout()
    _save_and_show(fig, f'13_shap_bar_{tag}.png')

    # Dependence plots: top-3 features
    top3 = feat_names[order[:3]]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, feat in zip(axes, top3):
        feat_idx = list(feat_names).index(feat)
        shap.dependence_plot(feat_idx, vals, X_bg, feature_names=feat_names, ax=ax, show=False, alpha=0.4)
        ax.set_title(feat, fontsize=9, fontweight='bold')
    fig.suptitle(f'SHAP Dependence Plots -- Top-3 Features ({model_name})', fontweight='bold', y=1.02)
    fig.tight_layout()
    _save_and_show(fig, f'14_shap_dependence_{tag}.png')

    return order, feat_names, mean_abs

print("SHAP utility defined.")

In [ ]:
# Reload models (in case running after a restart)
gbr   = joblib.load(os.path.join(MODELS_DIR, 'gbr.pkl'))
xgb_m = joblib.load(os.path.join(MODELS_DIR, 'xgb.pkl'))
lgb_m = joblib.load(os.path.join(MODELS_DIR, 'lgb.pkl'))
cb_m  = joblib.load(os.path.join(MODELS_DIR, 'cb.pkl'))
rf_m  = joblib.load(os.path.join(MODELS_DIR, 'rf.pkl'))
vot   = joblib.load(os.path.join(MODELS_DIR, 'voting.pkl'))

best_name = df_all[df_all['Split']=='Test'].loc[lambda x: x['R2'].idxmax(), 'Model']
print(f"Best model: {best_name}")

# Map result name -> loaded model object
model_map = {'GBR': gbr, 'XGBoost': xgb_m, 'LightGBM': lgb_m,
             'CatBoost': cb_m, 'RandomForest': rf_m, 'WeightedVoting': vot}

# SHAP for best model
order, feat_names, mean_abs = run_shap(model_map[best_name], best_name, X_test, feature_names, 'best')

# SHAP for WeightedVoting (if not already the best)
if best_name != 'WeightedVoting':
    print("\nAlso running SHAP for WeightedVoting ...")
    run_shap(vot, 'WeightedVoting', X_test, feature_names, 'voting')

In [ ]:
# ── Comparative importance across all 5 tree models ──────────────────────────
print("Building comparative importance chart ...")
tree_models = {'GBR': gbr, 'XGBoost': xgb_m, 'LightGBM': lgb_m, 'CatBoost': cb_m, 'RandomForest': rf_m}
imp_dict = {}
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=min(SHAP_SAMPLE, len(X_test)), replace=False)

for mn, m in tree_models.items():
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            exp = shap.TreeExplainer(m)
            sv  = exp.shap_values(X_test[idx])
        imp_dict[mn] = np.abs(sv).mean(axis=0)
    except Exception as e:
        print(f"  {mn} failed: {e}")

if imp_dict:
    df_imp = pd.DataFrame(imp_dict, index=feature_names)
    top_feats = df_imp.mean(axis=1).nlargest(15).index
    fig, ax = plt.subplots(figsize=(12, 7))
    df_imp.loc[top_feats].plot.barh(ax=ax, width=0.75, alpha=0.85)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Comparative Feature Importance -- All Tree Models (Top 15)', fontweight='bold')
    ax.invert_yaxis(); ax.legend(title='Model', loc='lower right')
    fig.tight_layout()
    _save_and_show(fig, '15_shap_comparative.png')

---
## Summary

All outputs are saved to the `outputs/` directory:

| Path | Contents |
|------|----------|
| `outputs/results/preprocessed.pkl` | Scaled train/val/test arrays + metadata DataFrames |
| `outputs/results/predictions.pkl` | Val/test predictions + CV scores for all 11 models |
| `outputs/results/metrics_table.csv` | Full metrics table (MSE, RMSE, MAE, R², Adj R², CV) |
| `outputs/models/` | Saved model files (`.pkl` for sklearn/xgb/lgb/cb, `.keras` for TF) |
| `outputs/figures/` | All 15 plots (EDA, metrics comparison, SHAP) |

### Key Results

| Model | Test MSE | Test R² | CV R² (mean±std) |
|-------|----------|---------|------------------|
| GBR | 0.0696 | 0.9996 | 0.9984 ± 0.0012 |
| Stacking | 0.0665 | 0.9996 | 0.9984 ± 0.0014 |
| WeightedVoting | 0.1079 | 0.9994 | 0.9958 ± 0.0048 |
| LightGBM | 0.1305 | 0.9993 | 0.9941 ± 0.0069 |
| LSTM | 5.0399 | 0.9725 | — |
| NN (MLP) | 12.5777 | 0.9315 | — |
| MLR | 19.5696 | 0.8935 | 0.8912 ± 0.0881 |

**SHAP top features** (consistent across all tree models): `pm2.5_avg`, `pm2.5`, `o3_8hr`
